# Entscheidungsbäume – Wer hätte den Untergang der Titanic überlebt?

In diesem Notebook lernen wir eine der **anschaulichsten** Methoden im Machine Learning kennen: den **Entscheidungsbaum** (engl. *Decision Tree*).

Als Datensatz nutzen wir die **Passagierliste der RMS Titanic** – echte Daten von 1912. Die Frage, die wir beantworten wollen, klingt wie aus einem Krimi:

> *Kann ein Computer allein anhand von Alter, Geschlecht, Klasse und Ticketpreis vorhersagen, ob ein Passagier die Katastrophe überlebt hat?*

Der Plan:
1. **Entscheidungsbäume** verstehen – wie funktionieren sie eigentlich?
2. **Titanic-Datensatz** laden und erkunden – was steckt in den Daten?
3. **Daten aufbereiten** – Texte in Zahlen umwandeln, fehlende Werte behandeln
4. **Modell trainieren** und den Baum anschauen – was hat der Computer gelernt?
5. **Eigene Schicksale vorhersagen** – hätten Rose und Jack überlebt?

## Was ist ein Entscheidungsbaum?

Stell dir vor, du möchtest entscheiden: **Soll ich heute draußen Sport machen?** Vermutlich gehst du in deinem Kopf eine Reihe von Fragen durch:

- *Regnet es?* → Wenn ja: drinnen bleiben.
- *Wenn nein: Habe ich Zeit?* → Wenn nein: drinnen bleiben.
- *Wenn ja: Sind meine Freunde dabei?* → Wenn ja: draußen Sport!

Das ist genau die Idee eines Entscheidungsbaums: ein **Baum aus Ja/Nein-Fragen**. Jede Frage spaltet die Daten in zwei Gruppen auf, bis am Ende eine klare **Entscheidung** übrig bleibt.

### Warum heißt das „Baum“?

Weil die Struktur aussieht wie ein **umgedrehter Baum**:

- Ganz oben ist die **Wurzel** – die erste, wichtigste Frage.
- Jede Frage erzeugt zwei **Äste** (Ja / Nein).
- Ganz unten sind die **Blätter** – dort steht das Ergebnis.

### Was macht den Algorithmus klug?

Der Computer **lernt von alleine**, welche Fragen er stellen soll und in welcher Reihenfolge. Er probiert verschiedene Fragen aus und wählt die, die die Daten am besten in Gruppen aufteilt. Das nennt man **Splitting**.

Für die Titanic wäre zum Beispiel eine gute erste Frage: *War der Passagier weiblich?* Denn die historische Regel *„Frauen und Kinder zuerst“* trennt die Daten sehr klar in zwei Gruppen mit sehr unterschiedlichen Überlebensraten.

### Wann sind Entscheidungsbäume besonders nützlich?

Im Vergleich zur **linearen Regression** (Notebook 04) und der **logistischen Regression** (Notebook 05) haben Entscheidungsbäume drei große Vorteile:

- **Man kann ihnen beim Denken zuschauen.** Wir können den Baum als Bild ausgeben und genau sehen, *warum* das Modell so entschieden hat. Keine Black Box!
- **Sie kommen mit Zahlen *und* Kategorien klar.** Geschlecht, Klasse, Hafen – alles kein Problem.
- **Sie brauchen keine Skalierung der Daten.** Pixel-Werte normieren wie in Notebook 05? Nicht nötig.

Der Nachteil: Ein einzelner Baum kann **überlernen** (engl. *overfitting*) – er merkt sich die Trainingsdaten zu genau und versagt bei neuen Daten. Wie man das vermeidet, lernen wir gleich.

---

## Teil 1: Der Titanic-Datensatz

Die **RMS Titanic** war zu ihrer Zeit das größte Schiff der Welt. In der Nacht vom **14. auf den 15. April 1912** kollidierte sie auf ihrer Jungfernfahrt mit einem Eisberg und sank.

Von den rund **2.224 Menschen** an Bord starben über **1.500**. Das lag auch daran, dass es nur Rettungsboote für etwa die Hälfte der Passagiere gab – und die Regel *„Frauen und Kinder zuerst“* angewandt wurde.

Die Passagierlisten sind erhalten geblieben und sind heute einer der berühmtesten Datensätze im Machine Learning. Wir laden ihn direkt aus scikit-learn:

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml

# Titanic-Datensatz laden (beim ersten Mal kann es ein paar Sekunden dauern)
titanic = fetch_openml("titanic", version=1, as_frame=True, parser="auto")
df = titanic.frame

print(f"Datensatz geladen: {df.shape[0]} Passagiere, {df.shape[1]} Spalten")

### Welche Informationen haben wir über jeden Passagier?

Schauen wir uns die ersten Zeilen an. Jede Zeile ist **ein Passagier**, jede Spalte eine Information über diese Person:

| Spalte | Bedeutung |
|---|---|
| `pclass` | Klasse des Tickets (1 = 1. Klasse, ..., 3 = 3. Klasse) |
| `survived` | **Zielvariable**: 1 = überlebt, 0 = gestorben |
| `name` | Vollständiger Name |
| `sex` | Geschlecht (`male` / `female`) |
| `age` | Alter in Jahren |
| `sibsp` | Anzahl Geschwister/Ehepartner an Bord |
| `parch` | Anzahl Eltern/Kinder an Bord |
| `fare` | Ticketpreis in Pfund |
| `embarked` | Einschiffungshafen (C = Cherbourg, Q = Queenstown, S = Southampton) |
| `cabin` | Kabinennummer (oft fehlend) |

In [ ]:
df.head()

In [ ]:
df.info()

---

## Teil 2: Daten erkunden – wer hat überlebt?

Bevor wir ein Modell trainieren, schauen wir uns die Daten genau an. **Was sagen die Zahlen über das Schicksal der Passagiere?**

### 2.1 – Wie viele überlebten insgesamt?

In [ ]:
# survived ist als String/Kategorie codiert – erst in Zahlen umwandeln
df["survived"] = df["survived"].astype(int)

ueberlebt = df["survived"].sum()
gesamt = len(df)
rate = ueberlebt / gesamt * 100

print(f"Passagiere im Datensatz:   {gesamt}")
print(f"davon überlebt:            {ueberlebt} ({rate:.1f}%)")
print(f"davon gestorben:           {gesamt - ueberlebt} ({100 - rate:.1f}%)")

### 2.2 – Überlebensrate nach Geschlecht

Stimmt die historische Regel *„Frauen und Kinder zuerst“* tatsächlich? Vergleichen wir die Überlebensraten von Männern und Frauen:

In [ ]:
rate_sex = df.groupby("sex")["survived"].mean() * 100

rate_sex.plot.bar(color=["coral", "steelblue"], edgecolor="white", figsize=(7, 4))
plt.title("Überlebensrate nach Geschlecht")
plt.ylabel("Überlebt (%)")
plt.xlabel("")
plt.xticks(rotation=0)
plt.ylim(0, 100)
plt.tight_layout()
plt.show()

for sex, r in rate_sex.items():
    print(f"   {sex:<8} → {r:.1f}% überlebten")

Ein klarer Unterschied! Frauen überlebten weit häufiger. Das ist ein **starkes Signal**, das unser Modell später ausnutzen wird.

### 2.3 – Überlebensrate nach Klasse

Hatte die Ticketklasse auch einen Einfluss? Die 1. Klasse war oben am Sonnendeck, die 3. Klasse tief unten im Schiff:

In [ ]:
rate_klasse = df.groupby("pclass")["survived"].mean() * 100

rate_klasse.plot.bar(color="seagreen", edgecolor="white", figsize=(7, 4))
plt.title("Überlebensrate nach Ticketklasse")
plt.ylabel("Überlebt (%)")
plt.xlabel("Klasse")
plt.xticks(rotation=0)
plt.ylim(0, 100)
plt.tight_layout()
plt.show()

for klasse, r in rate_klasse.items():
    print(f"   {klasse}. Klasse → {r:.1f}% überlebten")

### 2.4 – Wer war überhaupt an Bord? Altersverteilung

Wie alt waren die Passagiere? Ein **Histogramm** zeigt die Altersverteilung:

In [ ]:
alter = df["age"].dropna()

alter.plot.hist(bins=40, color="darkorange", edgecolor="white", figsize=(10, 4))
plt.title("Altersverteilung der Passagiere")
plt.xlabel("Alter (Jahre)")
plt.ylabel("Anzahl Passagiere")
plt.tight_layout()
plt.show()

print(f"Jüngste:r:          {alter.min():.1f} Jahre")
print(f"Älteste:r:          {alter.max():.1f} Jahre")
print(f"Durchschnittsalter: {alter.mean():.1f} Jahre")
print(f"Fehlende Angaben:   {df['age'].isna().sum()} Passagiere ohne Altersangabe")

---

## Teil 3: Daten aufbereiten

Bevor wir trainieren können, müssen wir die Daten **passend für das Modell** machen. Drei Dinge sind zu tun:

1. **Unwichtige Spalten entfernen.** Der Name oder die Ticketnummer hilft dem Modell nicht beim Vorhersagen.
2. **Fehlende Werte behandeln.** Bei vielen Passagieren fehlt das Alter. Wir ersetzen die Lücken mit dem **Median-Alter**.
3. **Text in Zahlen umwandeln.** scikit-learn kann mit Texten wie `"male"` oder `"female"` nicht direkt rechnen. Wir codieren sie als Zahlen.

In [ ]:
# 1. Spalten auswählen, die wir behalten
spalten = ["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked", "survived"]
daten = df[spalten].copy()

# 2. Fehlende Werte auffüllen
daten["age"]  = daten["age"].fillna(daten["age"].median())
daten["fare"] = daten["fare"].fillna(daten["fare"].median())
daten["embarked"] = daten["embarked"].fillna("S")  # häufigster Hafen

# 3. Kategorien in Zahlen umwandeln
daten["sex"]      = daten["sex"].map({"male": 0, "female": 1})
daten["embarked"] = daten["embarked"].map({"S": 0, "C": 1, "Q": 2})

# Alle Spalten als Zahlen sicherstellen (manche kommen aus openml als category)
daten = daten.astype(float)

print(f"Aufbereiteter Datensatz: {len(daten)} Zeilen, {daten.shape[1]} Spalten")
print(f"Fehlende Werte: {daten.isna().sum().sum()}\n")
daten.head()

---

## Teil 4: Train/Test-Split

Wie schon bei der Ziffernerkennung (Notebook 05) teilen wir die Daten auf:

- **Trainingsdaten** (80 %) – damit lernt das Modell
- **Testdaten** (20 %) – damit prüfen wir, wie gut es auf **neuen** Passagieren funktioniert

Wichtig: Das Modell darf die Testdaten beim Training **nicht** sehen – sonst könnten wir nicht ehrlich messen, wie gut es ist.

In [ ]:
from sklearn.model_selection import train_test_split

features = ["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]

X = daten[features]
y = daten["survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Trainingsdaten: {len(X_train)} Passagiere")
print(f"Testdaten:      {len(X_test)} Passagiere")

---

## Teil 5: Den Entscheidungsbaum trainieren

Jetzt kommt der spannende Teil! Wir benutzen `DecisionTreeClassifier` aus scikit-learn. Die wichtigsten Einstellungen:

- `max_depth=3` – der Baum darf maximal 3 Fragen tief werden. So bleibt er **klein und verständlich**.
- `random_state=42` – damit das Ergebnis reproduzierbar ist.

Später probieren wir auch tiefere Bäume aus.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

baum = DecisionTreeClassifier(max_depth=3, random_state=42)
baum.fit(X_train, y_train)

print("Entscheidungsbaum trainiert!")
print(f"Maximale Tiefe: {baum.get_depth()}")
print(f"Anzahl Blätter: {baum.get_n_leaves()}")

### Den Baum anschauen

Das Schönste an Entscheidungsbäumen: **Wir können sie als Bild sehen!** Jeder Knoten zeigt eine Frage, jeder Pfeil eine Antwort, jedes Blatt eine Vorhersage.

In [ ]:
from sklearn.tree import plot_tree

fig, ax = plt.subplots(figsize=(16, 8))
plot_tree(
    baum,
    feature_names=features,
    class_names=["gestorben", "überlebt"],
    filled=True,
    rounded=True,
    fontsize=10,
    ax=ax,
)
plt.title("Entscheidungsbaum für das Überleben auf der Titanic")
plt.tight_layout()
plt.show()

### Wie liest man so einen Baum?

Lies von **oben nach unten**. An jedem Knoten steht eine Frage – zum Beispiel `sex <= 0.5`. Erinnerung: Wir hatten `male = 0` und `female = 1` codiert. Die Frage heißt also auf Deutsch:

> *Ist der Passagier männlich?* (Wenn ja: links, wenn nein: rechts.)

Außerdem siehst du in jedem Knoten:

- `samples` – wie viele Passagiere in dieser Gruppe gelandet sind
- `value = [a, b]` – wie viele davon `gestorben` (a) und `überlebt` (b) sind
- `class` – die Mehrheitsklasse in dieser Gruppe (das würde der Baum als Vorhersage ausgeben)
- Die **Farbe** zeigt, wie eindeutig das Ergebnis ist: dunkelorange = überwiegend gestorben, dunkelblau = überwiegend überlebt.

Schau dir die linke Seite an (`sex <= 0.5` = männlich): überwiegend orange! Die meisten Männer sind gestorben. Auf der rechten Seite (weiblich): überwiegend blau – die meisten Frauen haben überlebt. Der Baum hat die Regel *„Frauen und Kinder zuerst“* praktisch selbst entdeckt!

### Wie gut ist unser Baum? – Accuracy

Erinnerung aus Notebook 05: Die **Accuracy** (Genauigkeit) gibt an, welcher Anteil der Vorhersagen richtig war.

In [ ]:
from sklearn.metrics import accuracy_score

vorhersagen = baum.predict(X_test)

acc_train = accuracy_score(y_train, baum.predict(X_train))
acc_test  = accuracy_score(y_test, vorhersagen)

print(f"Accuracy auf den Trainingsdaten: {acc_train:.4f}")
print(f"Accuracy auf den Testdaten:      {acc_test:.4f}")
print(f"\nDas Modell hat {(vorhersagen == y_test).sum()} von {len(y_test)} Passagieren richtig eingeordnet.")

---

## Zusammenfassung & Ausblick

Das haben wir bisher gelernt:

- Ein **Entscheidungsbaum** stellt eine Reihe von Ja/Nein-Fragen, bis er eine Vorhersage trifft.
- Der Computer lernt **von alleine**, welche Fragen die Daten am besten aufteilen (**Splitting**).
- Wir können den Baum als Bild ausgeben und **direkt nachvollziehen**, was er gelernt hat – keine Black Box!
- Für die Titanic hat der Baum praktisch die historische Regel *„Frauen und Kinder zuerst“* selbst entdeckt.

### Was kommt als nächstes?

In den Übungsaufgaben experimentierst du mit der **Tiefe** des Baums, findest heraus **welche Merkmale am wichtigsten sind**, und sagst sogar das Schicksal von **Rose und Jack** aus dem Film *Titanic* voraus.

---

# Übungsaufgaben – Entscheidungsbäume mit Titanic

Jetzt bist du dran! Du arbeitest mit dem Datensatz `daten` und den Train/Test-Splits `X_train`, `X_test`, `y_train`, `y_test` aus dem Erklär-Teil weiter.

### Aufgabe 1 (leicht): Einen eigenen Baum trainieren

Trainiere selbst einen Entscheidungsbaum – diesmal mit `max_depth=5`. Schau, ob er besser oder schlechter wird als der mit Tiefe 3.

**Schritt für Schritt:**

1. Erstelle einen neuen `DecisionTreeClassifier` mit `max_depth=5` und `random_state=42`.
2. Trainiere ihn mit `.fit()` auf `X_train` und `y_train`.
3. Berechne die Accuracy auf den **Trainingsdaten** und auf den **Testdaten**.
4. Vergleiche mit unserem Tiefe-3-Baum oben. Wird die Vorhersage auf den Testdaten wirklich besser?

In [ ]:
# Aufgabe 1: Baum mit max_depth=5

# 1. Modell erstellen
# mein_baum = DecisionTreeClassifier(max_depth=..., random_state=42)

# 2. Trainieren
# mein_baum.fit(...)

# 3. Accuracy berechnen
# acc_train = accuracy_score(y_train, mein_baum.predict(X_train))
# acc_test  = accuracy_score(y_test,  mein_baum.predict(X_test))

# print(f"Accuracy (Training): ...")
# print(f"Accuracy (Test):     ...")


### Aufgabe 2 (mittel): Overfitting sichtbar machen

Was passiert, wenn wir den Baum **immer tiefer** machen? Eine wichtige Beobachtung im Machine Learning:

> Ein zu tiefer Baum **merkt sich die Trainingsdaten auswendig**, aber er versagt bei neuen Daten. Das nennt man **Overfitting** (zu Deutsch: Überanpassung).

Stell dir vor, du lernst eine Matheprüfung, indem du jede einzelne Lösung auswendig lernst statt das Prinzip zu verstehen. Bei genau diesen Aufgaben bekommst du 100 % – aber bei neuen Aufgaben fällst du durch. Genau das passiert hier.

**Aufgabe:**

1. Trainiere für **jede** Tiefe von 1 bis 15 einen Baum.
2. Berechne für jeden Baum die Accuracy auf Training **und** Test.
3. Zeichne beide Kurven in **ein** Diagramm (x-Achse: `max_depth`, y-Achse: Accuracy).
4. Was beobachtest du? Bei welcher Tiefe ist der Test-Score am höchsten?

*Tipp:* Sammle die Accuracys in zwei Listen und plotte sie mit `plt.plot(tiefen, train_scores, label="Training")`.

In [ ]:
# Aufgabe 2: Accuracy in Abhängigkeit von max_depth

tiefen = range(1, 16)
train_scores = []
test_scores = []

for tiefe in tiefen:
    # b = DecisionTreeClassifier(...)
    # b.fit(...)
    # train_scores.append(...)
    # test_scores.append(...)
    pass

# Plot
# plt.plot(tiefen, train_scores, marker="o", color="steelblue", label="Training")
# plt.plot(tiefen, test_scores,  marker="o", color="darkorange", label="Test")
# plt.xlabel("max_depth")
# plt.ylabel("Accuracy")
# plt.title("Trainings- vs. Test-Accuracy nach Baumtiefe")
# plt.legend()
# plt.grid(alpha=0.3)
# plt.show()


### Aufgabe 3 (mittel): Welches Merkmal ist am wichtigsten?

Ein Entscheidungsbaum kann uns nicht nur Vorhersagen liefern, sondern auch **verraten, welche Merkmale am wichtigsten waren**. Das nennt man **Feature Importance**.

Je weiter oben (oder je häufiger) ein Merkmal im Baum verwendet wird und je sauberer es die Gruppen trennt, desto wichtiger ist es. scikit-learn berechnet das automatisch und du bekommst die Werte mit `baum.feature_importances_`.

**Aufgabe:**

1. Trainiere einen Baum mit der Tiefe, die in Aufgabe 2 am besten war.
2. Hole dir die `feature_importances_` und ordne sie den Spaltennamen `features` zu.
3. Sortiere die Merkmale nach Wichtigkeit.
4. Zeichne ein **horizontales Balkendiagramm** der Wichtigkeiten.
5. Welches Merkmal ist am wichtigsten? Entspricht das deiner Vermutung?

*Tipp:* Eine `pandas.Series` macht es einfach: `pd.Series(baum.feature_importances_, index=features).sort_values()`.

In [ ]:
# Aufgabe 3: Feature Importance

# 1. Baum mit der besten Tiefe aus Aufgabe 2 trainieren
# bester_baum = DecisionTreeClassifier(max_depth=..., random_state=42)
# bester_baum.fit(X_train, y_train)

# 2. + 3. Wichtigkeiten als sortierte Series
# wichtigkeit = pd.Series(bester_baum.feature_importances_, index=features).sort_values()

# 4. Horizontales Balkendiagramm
# wichtigkeit.plot.barh(color="seagreen", edgecolor="white", figsize=(8, 4))
# plt.title("Feature Importance – was hat der Baum als wichtig erkannt?")
# plt.xlabel("Wichtigkeit")
# plt.tight_layout()
# plt.show()

# 5. Welches Merkmal ist am wichtigsten?


### Aufgabe 4 (schwer): Hätten Rose und Jack überlebt?

Im Film *Titanic* von 1997 sind **Rose DeWitt Bukater** und **Jack Dawson** die Hauptfiguren. Wir bauen sie als Datenpunkte nach und lassen unseren Baum entscheiden, ob er ihnen das Filmende glaubt!

| Merkmal | Rose | Jack |
|---|---|---|
| `pclass` | 1 (1. Klasse) | 3 (3. Klasse) |
| `sex` | 1 (weiblich) | 0 (männlich) |
| `age` | 17 | 20 |
| `sibsp` | 0 | 0 |
| `parch` | 1 (Mutter dabei) | 0 |
| `fare` | 100 £ | 5 £ |
| `embarked` | 0 (Southampton) | 0 (Southampton) |

**Aufgabe:**

1. Erstelle einen kleinen DataFrame mit den zwei Personen (eine Zeile pro Person).
2. Nutze deinen besten Baum aus Aufgabe 3, um die Überlebenswahrscheinlichkeit für beide vorherzusagen.
3. Mit `.predict()` bekommst du die Klassen (0 oder 1), mit `.predict_proba()` die Wahrscheinlichkeiten für jede Klasse.
4. Gib für beide aus: Klasse (überlebt / gestorben) und die Wahrscheinlichkeit zu überleben in Prozent.
5. Passt das zum Film? *(Spoiler: Im Film stirbt Jack – obwohl Rose eigentlich problemlos zur Seite rücken könnte!)*

*Tipp:* Die Spaltenreihenfolge muss zu `features` passen – also `["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]`.

In [ ]:
# Aufgabe 4: Rose und Jack vorhersagen

# 1. DataFrame mit beiden Personen
# personen = pd.DataFrame(
#     [
#         [...],  # Rose
#         [...],  # Jack
#     ],
#     columns=features,
#     index=["Rose", "Jack"],
# )

# 2./3. Vorhersagen
# klassen = bester_baum.predict(personen)
# wahrsch = bester_baum.predict_proba(personen)

# 4. Ausgabe
# for name, klasse, wkt in zip(personen.index, klassen, wahrsch):
#     status = "überlebt" if klasse == 1 else "gestorben"
#     print(f"   {name}: {status}  (Überlebenswahrscheinlichkeit: {wkt[1]*100:.1f}%)")


### Profi-Aufgabe: Random Forest – ein ganzer Wald aus Bäumen

Ein einzelner Baum hat eine Schwachstelle: Er kann sich leicht verirren – je nachdem, welche Datenpunkte er zufällig zu sehen bekommt, entscheidet er anders.

Eine geniale Idee: **Bau einfach viele Bäume**, jeder mit einem zufälligen Ausschnitt der Daten, und lass sie **abstimmen**. Das nennt man **Random Forest** – einen „Zufallswald“. Es ist eine der beliebtesten Methoden im klassischen Machine Learning.

**Aufgabe:**

1. Importiere `RandomForestClassifier` aus `sklearn.ensemble`.
2. Trainiere einen Wald mit z. B. `n_estimators=200` (= 200 Bäume), `max_depth=5`, `random_state=42`.
3. Berechne die Test-Accuracy und vergleiche sie mit deinem besten einzelnen Baum aus Aufgabe 3.
4. Lass den Wald auch für Rose und Jack abstimmen. Ändert sich das Ergebnis?
5. **Bonusfrage:** Auch der Random Forest hat `feature_importances_`. Sind die wichtigsten Merkmale dieselben wie beim einzelnen Baum?

In [ ]:
# Profi-Aufgabe: Random Forest

# from sklearn.ensemble import RandomForestClassifier

# 1. + 2. Wald trainieren
# wald = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)
# wald.fit(...)

# 3. Vergleich Accuracy
# print(f"Bester Einzelbaum (Test): ...")
# print(f"Random Forest    (Test): ...")

# 4. Vorhersage für Rose und Jack
# wald_wahrsch = wald.predict_proba(personen)
# for name, wkt in zip(personen.index, wald_wahrsch):
#     print(f"   {name}: Überlebenswahrscheinlichkeit {wkt[1]*100:.1f}%")

# 5. Feature Importance des Waldes
# pd.Series(wald.feature_importances_, index=features).sort_values().plot.barh(
#     color="coral", edgecolor="white", figsize=(8, 4)
# )
# plt.title("Feature Importance – Random Forest")
# plt.tight_layout()
# plt.show()


---

## Geschafft!

Du hast heute gelernt:

- Wie ein **Entscheidungsbaum** Schritt für Schritt Fragen stellt, um eine Vorhersage zu treffen
- Wie man echte historische Daten (die Titanic-Passagierliste) **erkundet und aufbereitet**
- Wie man kategorische Werte (`male`/`female`, S/C/Q) in Zahlen umwandelt
- Wie man mit `plot_tree` einen Baum als Bild ausgibt und **liest**
- Was **Overfitting** ist und wie `max_depth` davor schützt
- Wie **Feature Importance** zeigt, welche Merkmale am wichtigsten waren
- Wie ein **Random Forest** aus vielen Bäumen oft noch besser wird als ein einzelner Baum

Die wichtigste Erkenntnis: Ein Entscheidungsbaum versteckt nichts. Du kannst seinen Baum anschauen und genau nachvollziehen, *warum* er so entschieden hat – das ist im Machine Learning ein großes Plus.